<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/use-cases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZeroGrad in six use cases

One notebook that walks the whole SDK on public data, from `pip install` to production feedback:

1. **Churn**: a classification championship on the IBM Telco customers.
2. **House prices**: a regression championship on California housing.
3. **Before you ship**: cross-validation, the threshold a missed churner justifies, and the columns that matter.
4. **Odd rows**: anomaly detection on the same customers, with a few broken records planted.
5. **Segments**: clustering wines with no labels.
6. **Production feedback** (optional, needs an API key): predictions on the platform, outcomes sent back, performance by cohort.

Everything up to step 5 runs locally, with no account. A GPU runtime makes it faster (*Runtime → Change runtime type → T4 GPU*), but CPU works; the samples are kept small so the whole notebook finishes in a few minutes either way. The first championship downloads each model's weights from its official repository, once.

Each step links to a longer notebook if you want to go deeper.

In [ ]:
!pip install -q zerograd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import zerograd
from zerograd import ZeroGrad

print("zerograd", zerograd.__version__)
zg = ZeroGrad(remote=False)   # local, even if ZEROGRAD_API_KEY is set; step 6 uses the platform
zerograd.list_models()

`list_models()` is the catalog: which task each model runs, its license and whether you can use it commercially. Non-commercial models stay off unless you ask for them by name.

## 1. Churn: who is about to leave?

The Telco customer table IBM publishes: one row per customer, with contract, services, charges and whether they left last month. We take 2,000 customers to learn from and keep 1,000 aside as "next month's" customers.

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # a few blanks for brand-new customers
telco = telco.sample(3_000, random_state=0)

history, upcoming = telco.iloc[:2_000], telco.iloc[2_000:]
print(telco.shape, "churn rate", round((telco["Churn"] == "Yes").mean(), 3))
telco.head()

`create` runs the championship: every installed model reads the same context, all of them are scored on the same holdout, and the best by ROC AUC becomes the champion. `customerID` is an identifier, so it goes in `drop`.

In [ ]:
churn = zg.create("telco-churn", examples=history, target="Churn", drop=["customerID"])
churn.evaluation

In [ ]:
churn.baselines   # the same holdout scored with no model, for scale

In [ ]:
churn.columns     # which columns were used, and why any were left out

The champion scores new customers: the predicted class, a score and the probability of each class.

In [ ]:
from sklearn.metrics import roc_auc_score

preds = churn.predict(upcoming.drop(columns=["Churn", "customerID"]))
print("ROC AUC on customers it never saw:", round(roc_auc_score(upcoming["Churn"] == "Yes", preds["proba_Yes"]), 3))
preds.head()

The same story in pictures, on the 1,000 customers the model never saw: how well the score ranks churners (ROC), how many of the flagged really leave at each recall (precision-recall), and whether a 30% score means 30% leave (calibration).

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

left = upcoming["Churn"] == "Yes"
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
RocCurveDisplay.from_predictions(left, preds["proba_Yes"], name=churn.champion_display, ax=axes[0], plot_chance_level=True)
PrecisionRecallDisplay.from_predictions(left, preds["proba_Yes"], name=churn.champion_display, ax=axes[1], plot_chance_level=True)
CalibrationDisplay.from_predictions(left, preds["proba_Yes"], n_bins=10, name=churn.champion_display, ax=axes[2])
axes[0].set_title("Who leaves first"); axes[1].set_title("Flags that are right"); axes[2].set_title("Scores you can read as odds")
plt.tight_layout()

In [ ]:
board = churn.evaluation["roc_auc"].sort_values()
ax = board.plot.barh(color=["#c9c3b8" if m != churn.champion_display else "#7c5cff" for m in board.index], figsize=(7, 2 + 0.4 * len(board)))
ax.axvline(0.5, color="gray", ls=":")
ax.text(0.505, -0.45, "no model", color="gray")
ax.set_xlim(0.4, 1); ax.set_ylabel(""); ax.set_xlabel("ROC AUC on the holdout"); ax.set_title("The podium");

### How much history does the champion need?

A foundation model reads its whole context on every prediction, so a smaller context answers faster. `context_curve` refits the champion on growing samples of its own context and scores each on the rows you pass; the recommended size is the smallest one within 0.005 of the best score. If it is smaller than what you have, `update(context_rows=...)` keeps that sample for predictions.

In [ ]:
curve = churn.context_curve(upcoming.drop(columns="customerID"), sizes=[100, 200, 400, 800, 1_600])
curve.plot(title="Churn: context size against ROC AUC");
curve

In [ ]:
print("recommended context:", curve.recommended, "rows")
churn.update(context_rows=curve.recommended)

In [ ]:
churn.save("telco-churn.zg")          # one small file, no weights inside
churn = zerograd.load("telco-churn.zg")
churn

More on the championship, one-row predictions and how much history you need: [quickstart](https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/quickstart.ipynb) and [Titanic](https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/titanic.ipynb).

## 2. House prices: regression

A numeric target with many distinct values is a regression, and the championship switches to RMSE, MAE and R². California housing ships with scikit-learn (it downloads once); the target is the median house value of a block, in units of $100,000.

In [ ]:
from sklearn.datasets import fetch_california_housing

houses = fetch_california_housing(as_frame=True).frame.sample(3_000, random_state=0)
train_h, test_h = houses.iloc[:2_000], houses.iloc[2_000:]

prices = zg.create("house-prices", examples=train_h, target="MedHouseVal")
prices.evaluation

In [ ]:
prices.baselines   # always the average and the median

In [ ]:
from sklearn.metrics import mean_absolute_error

guess = prices.predict(test_h.drop(columns="MedHouseVal"))
print("MAE on new blocks: $", round(100_000 * mean_absolute_error(test_h["MedHouseVal"], guess["prediction"])))
prices.predict(test_h.drop(columns="MedHouseVal").iloc[0].to_dict())   # a dict in, a dict out

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))
a.scatter(test_h["MedHouseVal"], guess["prediction"], s=6, alpha=0.4, color="#7c5cff")
lim = [0, max(test_h["MedHouseVal"].max(), guess["prediction"].max())]
a.plot(lim, lim, color="gray", ls=":"); a.set_xlabel("actual ($100K)"); a.set_ylabel("predicted ($100K)"); a.set_title("Predicted against actual")
(guess["prediction"] - test_h["MedHouseVal"].to_numpy()).plot.hist(bins=40, ax=b, color="#c9c3b8")
b.axvline(0, color="gray", ls=":"); b.set_xlabel("error ($100K)"); b.set_title("How far off");
plt.tight_layout()

## 3. Before you ship it

A single holdout is one draw. Before the churn model drives a retention campaign you want to know that it holds on other splits, where to cut the score, and what it is reading.

**Cross-validation.** Every model on 3 folds, next to a baseline, with KS and lift in the top 10% for binary targets. `groups=` and `time=` split by customer or by date when rows repeat or the model will predict the future; the [evaluation notebook](https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/evaluation.ipynb) shows both.

In [ ]:
cv = zg.cross_validate(history.drop(columns="customerID"), "Churn", folds=3)
cv.summary

**The threshold.** Say a churner you miss costs 5 times what a retention offer to a loyal customer costs. `optimize_threshold` reads the cut off the championship holdout, with no new predictions, and the capability predicts with it from then on.

In [ ]:
cut = churn.optimize_threshold("cost", cost_fp=1, cost_fn=5)
cut

In [ ]:
t = cut.table
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
t["cost"].plot(ax=a, color="#7c5cff"); a.axvline(cut.threshold, color="gray", ls=":")
a.set_title(f"Cost of the mistakes (best cut {cut.threshold:.2f})"); a.set_xlabel("threshold")
t[["precision", "recall"]].plot(ax=b); t["flagged"].plot(ax=b, ls="--", color="gray", label="flagged")
b.axvline(cut.threshold, color="gray", ls=":"); b.set_title("What each cut buys"); b.set_xlabel("threshold"); b.legend();
plt.tight_layout()

In [ ]:
from sklearn.metrics import confusion_matrix

flagged = churn.predict(upcoming.drop(columns=["Churn", "customerID"]))
print("threshold", round(churn.threshold, 3))
pd.DataFrame(confusion_matrix(upcoming["Churn"], flagged["prediction"], labels=["No", "Yes"]),
             index=["stayed", "left"], columns=["predicted stays", "predicted leaves"])

**The columns.** Each column is shuffled in rows the model never read, and the drop in ROC AUC says how much it carried.

In [ ]:
imp = churn.importance(upcoming.drop(columns="customerID"), repeats=2, max_rows=500)
ax = imp.head(10).iloc[::-1].plot.barh(y="importance", xerr="std", legend=False, color="#7c5cff", figsize=(8, 4),
                                        title="What the model reads (drop in ROC AUC when shuffled)")
ax.set_ylabel(""); plt.tight_layout()
imp

Prefer scikit-learn? `ZeroGradClassifier` and `ZeroGradRegressor` drop into pipelines, `cross_val_score` and grid searches:

In [ ]:
from sklearn.model_selection import cross_val_score

from zerograd import ZeroGradRegressor

X, y = houses.drop(columns="MedHouseVal").iloc[:1_500], houses["MedHouseVal"].iloc[:1_500]
cross_val_score(ZeroGradRegressor(model="tabicl-v2"), X, y, cv=3, scoring="r2").round(3)

## 4. Odd rows: anomaly detection

No target this time. We break 15 customers on purpose (two years of tenure billed like twenty, or a monthly charge no plan has) and ask which rows look wrong. With `labels=` the planted rows score every method, foundation models next to Isolation Forest, LOF and kNN; the label is never read as a feature.

In [ ]:
rng = np.random.default_rng(0)
cols = ["tenure", "MonthlyCharges", "TotalCharges", "Contract", "InternetService", "PaymentMethod"]
book = history[cols].dropna().iloc[:800].reset_index(drop=True)
book["broken"] = 0
bad = rng.choice(len(book), 15, replace=False)
book.loc[bad[:8], "TotalCharges"] *= 10
book.loc[bad[8:], "MonthlyCharges"] = rng.uniform(400, 600, 7).round(2)
book.loc[bad, "broken"] = 1

odd = zg.anomalies(book, labels="broken")
odd.evaluation

Without labels, the default model scores every row, and TabPFN v2 and TabICLv2 also say which columns make it unusual, in `why`:

In [ ]:
blind = zg.anomalies(book.drop(columns="broken"), models=["tabpfn-v2"])
print("planted rows flagged:", int(blind.scores.loc[bad, "is_anomaly"].sum()), "of", len(bad))
blind.top(10)

In [ ]:
scores = blind.scores["anomaly_score"]
ax = scores.plot.hist(bins=50, color="#c9c3b8", figsize=(8, 3.5), label="all customers")
ax.scatter(scores.loc[bad], np.zeros(len(bad)), color="#e4572e", zorder=3, marker="^", s=60, label="the broken ones")
ax.set_xlabel("anomaly score"); ax.set_title("Broken rows land on the right"); ax.legend();

## 5. Segments: clustering

The wine dataset in scikit-learn: 178 wines, 13 chemical measurements, three cultivars. ZEUS, a zero-shot clustering model, competes with k-means and a Gaussian mixture; the cultivar is used only to score them (adjusted Rand index). Leave `labels` out and the SDK picks `k` for you.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True).frame
segments = zg.cluster(wine, labels="target")
segments.evaluation

In [ ]:
segments.profiles   # what sets each segment apart

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

xy = PCA(2).fit_transform(StandardScaler().fit_transform(wine.drop(columns="target")))
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
a.scatter(*xy.T, c=segments.labels, cmap="viridis", s=18); a.set_title(f"Segments found ({segments.model}, no labels used)")
b.scatter(*xy.T, c=wine["target"], cmap="viridis", s=18); b.set_title("The real cultivars")
for ax in (a, b):
    ax.set_xticks([]); ax.set_yticks([])

Imputation and synthetic rows are in the [without a target notebook](https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/no-target.ipynb).

## 6. Production feedback on the platform (optional)

This step needs a ZeroGrad API key. In Colab, add it as a secret named `ZEROGRAD_API_KEY` (the key icon on the left) and give this notebook access; anywhere else, set the environment variable. Without a key the cells below skip themselves.

On the platform every prediction gets a `prediction_id`, and with `id_column` your own id is kept as its reference. When the real outcomes arrive you send them back, and `performance` scores each cohort of predictions next to the holdout. Here the "future" outcomes are the ones we held aside, so you can see the loop end to end in a minute. `maturity_days=0` makes every cohort count right away; in real life you would set it to how long an outcome takes to be known (say 30 days for churn).

Before anything is uploaded, the SDK drops columns that look like personal data on your side and warns you about it.

In [ ]:
import os

key = os.environ.get("ZEROGRAD_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("ZEROGRAD_API_KEY")
    except Exception:
        key = None
print("API key found: running step 6" if key else "No API key: step 6 is skipped")

In [ ]:
if key:
    cloud = ZeroGrad(api_key=key, remote=True)
    remote_churn = cloud.create("telco-churn-demo", examples=history.drop(columns="customerID"), target="Churn")
    display(remote_churn.evaluation)

In [ ]:
if key:
    live = remote_churn.predict(upcoming.drop(columns="Churn"), id_column="customerID")
    display(live.head())

In [ ]:
if key:
    outcomes = pd.DataFrame({"reference": upcoming["customerID"], "outcome": upcoming["Churn"]})
    print(remote_churn.send_outcomes(outcomes))
    remote_churn.update(maturity_days=0)
    perf = remote_churn.performance(cohort="day")
    display(perf)
    print("overall:", perf.attrs["overall"])

In [ ]:
if key:
    remote_churn.delete()   # clean up the demo capability; leave this out to keep it in your workspace

## Where to go next

* Pick the competitors: `ZeroGrad(models=["tabpfn-v2", "mitra-v2"])` (Mitra-v2 and TabDPT come with `pip install "zerograd[all]"`).
* Licenses: TabPFN v2 asks products built with it to show *Built with PriorLabs-TabPFN*. See [MODELS.md](https://github.com/zerogradlabs/zerograd-python/blob/main/MODELS.md) and [TERMS.md](https://github.com/zerogradlabs/zerograd-python/blob/main/TERMS.md).
* Docs: [zerograd.dev/docs](https://zerograd.dev/docs).